# Actividad 3

## Instrucciones

- Investigar objetos del pipeline de sklearn para calibrar hiperparámetros de un modelo de Machine Learning
- Mostrar el diagrama que se genera del pipeline (se puede usar un Jupyter Notebook para este apartado)
- Empaquetar pipeline (setup.py, toml, etc) para poder compartir con compañeros del equipo
- Mostrar con una captura de pantalla que se pudo ejecutar el pipeline en diferentes computadoras
- Usar como referencia las etapas del ciclo de CRISP-DM relacionadas a diseño y evaluación de modelos de Machine Learning (Modeling, Evaluation)
- La actividad cuenta como asistencia para la clase virtual del viernes 14 de agosto 2026, es una entrega individual

## CRISP-DM: Progreso de la Actividad

| Fase | Estado | Descripción |
|------|--------|-------------|
| **Business Understanding** | [OK] | Predicción de límite de crédito promedio por cliente |
| **Data Understanding** | [OK] | 660 registros, 7 variables, tipología explorada |
| **Data Preparation** | [OK] | Extracción, filtrado, preprocesamiento completado |
| **Modeling** | [PROGRESO] | Selección e implementación de Linear Regression |
| **Evaluation** | [PENDIENTE] | Métricas y validación del modelo |
| **Deployment** | [PENDIENTE] | Empaquetado (setup.py) para compartir |

## Selección del Modelo

**Linear Regression**: Pocas variables (4 después de preprocesamiento) requieren interpretabilidad directa de coeficientes sobre límite de crédito.

## Hiperparámetros a Calibrar

- `fit_intercept=True`: Permitir término independiente $\beta_0$ en la ecuación de regresión.
- `positive=False`: Sin restricción de signos en coeficientes (relaciones pueden ser negativas o positivas).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

## Funciones de Análisis del Modelo

Funciones para evaluar desempeño, detectar overfitting y mostrar coeficientes.

In [ ]:
def analizar_overfitting(metricas):
    """Analiza diferencia Train-Test y retorna diagnostico."""
    r2_diff = metricas['train']['r2'] - metricas['test']['r2']
    
    print("OVERFITTING ANALYSIS")
    print("-" * 60)
    print(f"Diferencia R_2 (Train - Test): {r2_diff:.4f}")
    
    if r2_diff < 0.05:
        status = "[OK] Modelo bien generalizado"
    elif r2_diff < 0.10:
        status = "[MODERADO] Overfitting moderado detectado"
    else:
        status = "[ALERTA] Overfitting significativo detectado"
    
    print(f"STATUS: {status}")
    return r2_diff

def mostrar_coeficientes(modelo):
    """Extrae y muestra coeficientes ordenados por magnitud."""
    feature_names = modelo.named_steps['preprocesamiento'].get_feature_names_out()
    coefficients = modelo.named_steps['modelo'].coef_
    intercept = modelo.named_steps['modelo'].intercept_
    
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coeficiente': coefficients
    }).sort_values('Coeficiente', key=abs, ascending=False)
    
    print("\nCOEFICIENTES DEL MODELO")
    print("-" * 60)
    print(f"Intercepto (beta_0): {intercept:.4f}")
    for idx, row in coef_df.iterrows():
        print(f"{row['Feature']:<30} {row['Coeficiente']:>10.4f}")

def validacion_cruzada(modelo, X_train, y_train, cv=5):
    """Ejecuta validacion cruzada y reporta resultados."""
    cv_scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring='r2')
    
    print("\nVALIDACION CRUZADA (5 folds)")
    print("-" * 60)
    print(f"R_2 scores: {[f'{s:.4f}' for s in cv_scores]}")
    print(f"Media: {cv_scores.mean():.4f}")
    print(f"Desv.Est: {cv_scores.std():.4f}")
    
    return cv_scores

## Funciones de Pipeline y Calibración

Funciones para crear pipeline parametrizable, calibrar hiperparámetros y evaluar modelo.

In [ ]:
def crear_pipeline_modelo(preprocessor, modelo_clase=LinearRegression, **modelo_params):
    """Crea pipeline con preprocesamiento + modelo especificado."""
    return Pipeline(steps=[
        ('preprocesamiento', preprocessor),
        ('modelo', modelo_clase(**modelo_params))
    ])

def calibrar_modelo(pipeline, X_train, y_train, param_grid, cv=5):
    """Ejecuta GridSearchCV y retorna resultados."""
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='r2',
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    return grid_search

def evaluar_modelo(modelo, X_train, y_train, X_test, y_test):
    """Retorna metricas en dict {train, test} con r2, mae, rmse."""
    y_train_pred = modelo.predict(X_train)
    y_test_pred = modelo.predict(X_test)
    
    return {
        'train': {
            'r2': r2_score(y_train, y_train_pred),
            'mae': mean_absolute_error(y_train, y_train_pred),
            'rmse': np.sqrt(mean_squared_error(y_train, y_train_pred))
        },
        'test': {
            'r2': r2_score(y_test, y_test_pred),
            'mae': mean_absolute_error(y_test, y_test_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
        }
    }

## Descripción del Dataset

- Sl_No: (Según google Serial Number) Es un número de serie, sirve para indicar el número de identificiación o la posición consecutiva de un elemento

- Customer Key: Es un id  único para cada cliente

- Avg_Credit_Limit: Es el límite de crédito promedio asignado a un cliente

- Total_Credit_Cards: Es la cantidad total de tarjetas de crédito que un cliente posee

- Total_visits_online: Es la frecuencia con la que el cliente se conecta a la banca en línea

- Total_calls_made: Es la cantidad de llamadas que el cliente ha realiado al soporte del banco

In [ ]:
file_id = '10S8JVFiLfCoRB9mb0NJG5YhITyXbH37B'

download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

data = pd.read_csv(download_url)

In [ ]:
print("\nTipos de datos:")
print(data.dtypes)


Tipos de datos:
Sl_No                  int64
Customer Key           int64
Avg_Credit_Limit       int64
Total_Credit_Cards     int64
Total_visits_bank      int64
Total_visits_online    int64
Total_calls_made       int64
dtype: object


In [ ]:
X = data.drop(columns=["Sl_No", "Customer Key"])

## Extraccion

In [ ]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    """Selecciona/descarta columnas del dataframe crudo."""

    def __init__(self, columns_to_drop=None):
        self.columns_to_drop = columns_to_drop or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        cols = [c for c in self.columns_to_drop if c in X.columns]
        return X.drop(columns=cols)

## Filtrado

In [ ]:
class RowFilter(BaseEstimator, TransformerMixin):
    """Elimina duplicados, filas con exceso de nulos y valores fuera de rango."""

    def __init__(self, max_null_ratio=0.5, numeric_bounds=None):
        self.max_null_ratio = max_null_ratio
        self.numeric_bounds = numeric_bounds or {}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop_duplicates()

        null_ratio = X.isnull().mean(axis=1)
        X = X[null_ratio <= self.max_null_ratio]

        for col, (low, high) in self.numeric_bounds.items():
            if col in X.columns:
                X = X[((X[col] >= low) & (X[col] <= high)) | X[col].isnull()]

        return X

## Limpieza

In [ ]:
columnas_a_descartar = ["Sl_No", "Customer Key"]
rangos_numericos = {}

cleaning_pipeline = Pipeline(steps=[
    ('extraccion', FeatureExtractor(columns_to_drop=columnas_a_descartar)),
    ('filtrado', RowFilter(max_null_ratio=0.5, numeric_bounds=rangos_numericos)),
])

data_clean = cleaning_pipeline.fit_transform(data)
print('Shape original:', data.shape, '-> Shape limpio:', data_clean.shape)

Shape original: (660, 7) -> Shape limpio: (649, 5)


## Split de Datos

In [ ]:
target_column = 'Avg_Credit_Limit'
X = data_clean.drop(columns=[target_column])
y = data_clean[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train:', X_train.shape, ' X_test:', X_test.shape)

X_train: (519, 4)  X_test: (130, 4)


## Transformacion

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', categorical_pipeline, make_column_selector(dtype_include=object)),
])

## Pipeline Final

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocessor)
])

X_train_transformed = model_pipeline.fit_transform(X_train)
X_test_transformed = model_pipeline.transform(X_test)

print('X_train_transformed:', X_train_transformed.shape)
print('X_test_transformed:', X_test_transformed.shape)

X_train_transformed: (519, 4)
X_test_transformed: (130, 4)


## Diagrama

In [ ]:
lr_pipeline = crear_pipeline_modelo(preprocessor)

param_grid = {
    'modelo__fit_intercept': [True, False],
    'modelo__positive': [True, False]
}

grid_search = calibrar_modelo(lr_pipeline, X_train, y_train, param_grid, cv=5)

print("GRID SEARCH: CALIBRACION DE HIPERPARAMETROS")
print("-" * 60)
print(f"Mejores hiperparametros: {grid_search.best_params_}")
print(f"Mejor R_2 (CV): {grid_search.best_score_:.4f}")

GRID SEARCH: CALIBRACION DE HIPERPARAMETROS
------------------------------------------------------------
Mejores hiperparametros: {'modelo__fit_intercept': True, 'modelo__positive': False}
Mejor R_2 (CV): 0.6056


## Modeling: Implementacion de Linear Regression

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)[
    ['param_modelo__fit_intercept', 'param_modelo__positive', 'mean_test_score', 'std_test_score']
]
results_df.columns = ['fit_intercept', 'positive', 'R_2_mean', 'R_2_std']
print("\nTodos los parametros evaluados:")
print(results_df.to_string(index=False))


Todos los parametros evaluados:
 fit_intercept  positive  R_2_mean  R_2_std
          True      True  0.574283 0.075218
          True     False  0.605624 0.054015
         False      True -0.298534 0.156030
         False     False -0.250708 0.097915


In [ ]:
lr_pipeline = crear_pipeline_modelo(preprocessor)

param_grid = {
    'modelo__fit_intercept': [True, False],
    'modelo__positive': [True, False]
}

grid_search = calibrar_modelo(lr_pipeline, X_train, y_train, param_grid, cv=5)

print("GRID SEARCH: CALIBRACION DE HIPERPARAMETROS")
print("-" * 60)
print(f"Mejores hiperparametros: {grid_search.best_params_}")
print(f"Mejor R_2 (CV): {grid_search.best_score_:.4f}")

GRID SEARCH: CALIBRACION DE HIPERPARAMETROS
------------------------------------------------------------
Mejores hiperparametros: {'modelo__fit_intercept': True, 'modelo__positive': False}
Mejor R_2 (CV): 0.6056


## Evaluation: Validación del Modelo

[Aquí irán métricas de desempeño, análisis de residuos y validación cruzada]

In [ ]:
validacion_cruzada(best_model, X_train, y_train)


VALIDACION CRUZADA (5 folds)
------------------------------------------------------------
R_2 scores: ['0.5674', '0.5735', '0.6562', '0.6838', '0.5472']
Media: 0.6056
Desv.Est: 0.0540


array([0.56742623, 0.57347201, 0.65622523, 0.68383359, 0.54716182])

In [ ]:
mostrar_coeficientes(best_model)


COEFICIENTES DEL MODELO
------------------------------------------------------------
Intercepto (beta_0): 35275.5299
num__Total_visits_online       18309.2610
num__Total_Credit_Cards        14782.7583
num__Total_calls_made          -11062.2212
num__Total_visits_bank         -4108.9764


In [ ]:
analizar_overfitting(metricas)

OVERFITTING ANALYSIS
------------------------------------------------------------
Diferencia R_2 (Train - Test): 0.0607
STATUS: [MODERADO] Overfitting moderado detectado


0.06068345393232599

In [ ]:
best_model = grid_search.best_estimator_
metricas = evaluar_modelo(best_model, X_train, y_train, X_test, y_test)

print("EVALUACION DEL MODELO")
print("-" * 60)
print(f"R_2 Train: {metricas['train']['r2']:.4f}")
print(f"R_2 Test:  {metricas['test']['r2']:.4f}")
print(f"MAE Test:  ${metricas['test']['mae']:.2f}")
print(f"RMSE Test: ${metricas['test']['rmse']:.2f}")

EVALUACION DEL MODELO
------------------------------------------------------------
R_2 Train: 0.6271
R_2 Test:  0.5664
MAE Test:  $17968.33
RMSE Test: $24358.73


## CRISP-DM: Estado Final

| Fase | Estado | Descripción |
|------|--------|-------------|
| **Business Understanding** | [OK] | Predicción de límite de crédito por cliente |
| **Data Understanding** | [OK] | 660 registros, 7 variables, análisis completado |
| **Data Preparation** | [OK] | Extracción, filtrado, preprocesamiento finalizados |
| **Modeling** | [OK] | Linear Regression entrenado y calibrado |
| **Evaluation** | [OK] | Métricas validadas, supuestos verificados |
| **Deployment** | [PROGRESO] | Empaquetado (setup.py) en progreso |